In [1]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


# Read in person cohort csv

In [2]:
person <- read.csv('data/additional_person_cohorts.csv', header = TRUE)

In [7]:
head(person)

,person_id,birth_date,gender,CombinedEthnicity,cohort
,<chr>,<chr>,<chr>,<chr>,<chr>
1,EE91E3C77F2B7CB44F1F601317E2DE5EE76ABB0941F24D4714026B3E30FC313C,1998-09-15,M,South Asian,1998/99
2,C5553DE427E9B8F60E9B5D1E83391E1B660F415A3924B630742D52A0517F5219,1998-09-15,M,White British,1998/99
3,AC8CB4071E24E606BFFECC380A9EA11546158370916C4D8D954FA76D63FA81AE,1998-09-15,M,White British,1998/99
4,E4E84D0DEB6692712A8E875B701E097EB61235069C56626900C0951E9B112B6D,1998-09-15,M,South Asian,1998/99
5,64FA4602D88C275E27FDBD17EA4678E378D7B297D91C2CDB914150ED0CBF3785,1998-09-15,M,White British,1998/99
6,F516562D5CFDDC6CD66FB8EA18399D11BB0E2E042F569D918285FF96C3377338,1998-09-15,M,White British,1998/99


In [8]:
person |>
nrow()

[1] 23385

In [3]:
person_ids <- read.csv('data/additional_person_ids.csv', header = TRUE)

In [10]:
head(person_ids)

,person_id,NCCIS_ACADYR
,<chr>,<chr>
1,0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2016/2017
2,00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2015/2016
3,000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,2015/2016
4,00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,2016/2017
5,0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2016/2017
6,0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2016/2017


In [11]:
person_ids |>
nrow()

[1] 18999

In [12]:
# Check if all person_ids in person_id are in person
all(person_ids$person_id %in% person$person_id)

[1] TRUE

In [13]:
# filter person to only matching IDs
person_filtered <- person |>
    filter(person_id %in% person_ids$person_id)

In [14]:
person_filtered |>
nrow()

[1] 17538

In [15]:
all(person_ids$person_id %in% person_filtered$person_id)

[1] TRUE

# Siblings / home indicators

In [16]:
school_census_data = "CB_2489.cb_StudentAdditionalInfoFromCensus"

In [17]:
home_table <- tbl(con, school_census_data) |>
    select(person_id, SourceTable, DistCurrSch, Language, IDACIScore_Home, LSOA11_Home, InCare) |>
    filter(!is.na(person_id))

In [188]:
# SGA means based on shared address (vs. on address and surname)
# BirthOrder_SGA 1 is the oldest

In [18]:
home_table

# Source:   SQL [?? x 7]
# Database: BigQueryConnection
   person_id SourceTable DistCurrSch Language IDACIScore_Home LSOA11_Home InCare
   <chr>     <chr>             <dbl> <chr>              <dbl> <chr>       <chr> 
 1 1F8E5FB8… Spring_Cen…        0.48 KUR                   NA E01010625   NA    
 2 677579B6… Spring_Cen…        0.48 RMN                   NA E01010620   NA    
 3 48384998… Spring_Cen…        1.74 ARA                   NA E01010840   NA    
 4 53AF9BA8… Spring_Cen…        2.06 CZE                   NA E01013567   NA    
 5 AEFA5D1B… Spring_Cen…        0.08 PRSD                  NA E01010872   NA    
 6 CA7DE3B3… Spring_Cen…        1.45 ARAS                  NA E01010723   NA    
 7 A869AB84… Spring_Cen…        0.22 ENG                   NA E01033258   NA    
 8 A04AE770… Spring_Cen…        1.96 ENG                   NA E01010845   NA    
 9 1E6FB869… Spring_Cen…        0.19 ENG                   NA E01010717   NA    
10 9C16DB10… Spring_Cen…        0.36 ENG             

In [19]:
home_table_df <- home_table |>
    collect()

In [20]:
home_table_df |>
    nrow()

[1] 6656974

In [21]:
home_table_df |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
302369


In [22]:
home_table_filtered <- home_table_df |>
    filter(person_id %in% person_ids$person_id)

In [23]:
home_table_filtered |>
    nrow()

[1] 546604

In [24]:
home_table_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17403


In [25]:
# extract year from source table
home_table_filtered$Year <- substr(home_table_filtered$SourceTable, nchar(home_table_filtered$SourceTable) - 3, nchar(home_table_filtered$SourceTable))

In [26]:
head(home_table_filtered)

person_id,SourceTable,DistCurrSch,Language,IDACIScore_Home,LSOA11_Home,InCare,Year
<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>
2FA132D5DC56E77DE5E30728AA576966E7F2008CB3F96E1BA9F8754FD5F8DEC3,PLASC_2004,0.13,NA,0.37225,NA,0,2004
391B645B52F18FFE649DD81418E83E190381418622EDD73A2831C9BEE36DC132,PLASC_2004,0.14,NA,0.19546,NA,0,2004
164E53437CFA00EF9777758F6F076C95830196AAEDBCF99D8A9418ED78112FB5,Spring_Census_2007,0.26,ENG,0.09306,NA,0,2007
0BE8A6BB7BC4E2EF84D250DBB3D085F543206E0397395BFED5B6FA9FD2455DFA,Spring_Census_2007,0.31,SPA,0.47297,NA,0,2007
018B1258C2F1AEE4098F69728BCD61326D343AC119B826BE21430E8D7DAD2857,Spring_Census_2012,0.84,ENG,0.15428,NA,NA,2012
509F6B8CF341FB222E705C40C234F9AE8011298F6BC699B32A9E99D59A8F9D14,Autumn_Census_2008,NA,ENG,0.42181,NA,NA,2008


In [27]:
# drop source table col and then merge / remove duplicate rows
home_table_filtered <- home_table_filtered |>
    select(-SourceTable)

In [28]:
home_table_filtered <- home_table_filtered |>
    distinct()

In [29]:
home_table_filtered |>
    nrow()

[1] 457722

In [30]:
home_table_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17403


#### drop cols

In [33]:
head(home_table_filtered)

person_id,DistCurrSch,Language,IDACIScore_Home,LSOA11_Home,Year
<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>
2FA132D5DC56E77DE5E30728AA576966E7F2008CB3F96E1BA9F8754FD5F8DEC3,0.13,NA,0.37225,NA,2004
391B645B52F18FFE649DD81418E83E190381418622EDD73A2831C9BEE36DC132,0.14,NA,0.19546,NA,2004
164E53437CFA00EF9777758F6F076C95830196AAEDBCF99D8A9418ED78112FB5,0.26,ENG,0.09306,NA,2007
0BE8A6BB7BC4E2EF84D250DBB3D085F543206E0397395BFED5B6FA9FD2455DFA,0.31,SPA,0.47297,NA,2007
018B1258C2F1AEE4098F69728BCD61326D343AC119B826BE21430E8D7DAD2857,0.84,ENG,0.15428,NA,2012
509F6B8CF341FB222E705C40C234F9AE8011298F6BC699B32A9E99D59A8F9D14,NA,ENG,0.42181,NA,2008


In [32]:
home_table_filtered <- home_table_filtered |>
    select(-InCare)

In [34]:
# add in cohort identifier so that can drop data from primary school ranges
home_table_merge <- home_table_filtered |>
  left_join(person_filtered |> select(person_id, cohort), by = "person_id")

In [35]:
home_table_merge |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17403


In [37]:
head(home_table_merge)

person_id,DistCurrSch,Language,IDACIScore_Home,LSOA11_Home,Year,cohort
<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>
2FA132D5DC56E77DE5E30728AA576966E7F2008CB3F96E1BA9F8754FD5F8DEC3,0.13,NA,0.37225,NA,2004,1998/99
391B645B52F18FFE649DD81418E83E190381418622EDD73A2831C9BEE36DC132,0.14,NA,0.19546,NA,2004,1998/99
164E53437CFA00EF9777758F6F076C95830196AAEDBCF99D8A9418ED78112FB5,0.26,ENG,0.09306,NA,2007,1998/99
0BE8A6BB7BC4E2EF84D250DBB3D085F543206E0397395BFED5B6FA9FD2455DFA,0.31,SPA,0.47297,NA,2007,1998/99
018B1258C2F1AEE4098F69728BCD61326D343AC119B826BE21430E8D7DAD2857,0.84,ENG,0.15428,NA,2012,1999/00
509F6B8CF341FB222E705C40C234F9AE8011298F6BC699B32A9E99D59A8F9D14,NA,ENG,0.42181,NA,2008,1998/99


In [39]:
home_table_merge <- home_table_merge |> 
    filter(Year %in% c('2010', '2011', '2012','2013','2014','2015'))

In [40]:
home_table_merge |> distinct(Year) |> pull(Year)

[1] "2012" "2013" "2014" "2010" "2015" "2011"

In [41]:
# where cohort 2001/2 drop 2013 year records
home_table_merge <- home_table_merge |> 
    filter(!(cohort == '1999/00' & Year == '2010'))

In [42]:
# where cohort 2000/1 drop 2018 year records
home_table_merge <- home_table_merge |> 
    filter(!(cohort == '1998/99' & Year == '2015'))

In [43]:
home_table_merge |>
    nrow()

[1] 209168

In [44]:
home_table_merge |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17208


In [45]:
# drop rows per year with NAs
home_table_merge <- home_table_merge |>
    arrange(desc(Year)) |>
    group_by(person_id, Year) |>
    summarise(across(everything(), ~ ifelse(all(is.na(.)), NA, na.omit(.)[1])), .groups = "drop")

In [46]:
home_table_merge |>
    nrow()

[1] 83043

In [47]:
home_table_merge |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17208


In [221]:
home_table_merge |>
    filter(person_id == '0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2014,1.59,0.81,ENG,0.20557,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2015,1.59,0.48,ENG,0.20557,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2016,1.59,0.48,ENG,NA,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2017,7.01,0.00,ENG,0.12500,E01002563,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,7.01,0.00,ENG,NA,E01002563,2001/02


In [48]:
# group by all cols except year, and keep first row of duplicates
home_table_merge <- home_table_merge |>
    arrange(desc(Year)) |>
    group_by(across(-Year)) |>
    filter(row_number() == 1) |>
    ungroup()

In [49]:
home_table_merge |>
    nrow()

[1] 64037

In [225]:
home_table_merge |>
    filter(person_id == '0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,7.01,0.00,ENG,NA,E01002563,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2017,7.01,0.00,ENG,0.12500,E01002563,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2016,1.59,0.48,ENG,NA,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2015,1.59,0.48,ENG,0.20557,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2014,1.59,0.81,ENG,0.20557,E01002616,2001/02


In [50]:
# repeat merge rows where there are NA values keep first non NA (latest date prioritised)
home_table_merge <- home_table_merge |>
    arrange(desc(Year)) |>
    group_by(person_id) |>
    summarise(across(everything(), ~ ifelse(all(is.na(.)), NA, na.omit(.)[1])), .groups = "drop")

In [51]:
head(home_table_merge)

person_id,Year,DistCurrSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2015,2.45,ENG,0.04778,E01027555,1999/00
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2014,4.71,ENG,0.35519,E01010703,1998/99
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,2014,0.33,ENG,0.37575,E01010742,1998/99
00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,2015,1.78,ENG,0.07161,E01010641,1999/00
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2015,0.76,ENG,0.44704,E01010655,1999/00
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2015,1.29,ENG,0.41032,E01010626,1999/00


In [229]:
home_table_merge |>
    filter(person_id == '0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,7.01,0,ENG,0.125,E01002563,2001/02


In [52]:
home_table_merge |> nrow()

[1] 17208

In [53]:
home_table_merge |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17208


#### EAL 

In [54]:
home_table_merge <- home_table_merge |>
    mutate(
    EAL = case_when(
      Language == "ENG" ~ 0,
      is.na(Language) ~ NA,
      TRUE ~ 1 
    )
  )  

In [55]:
home_table_merge <- home_table_merge |>
    select(-c(Language, Year))

In [56]:
head(home_table_merge)

person_id,DistCurrSch,IDACIScore_Home,LSOA11_Home,cohort,EAL
<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2.45,0.04778,E01027555,1999/00,0
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,4.71,0.35519,E01010703,1998/99,0
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,0.33,0.37575,E01010742,1998/99,0
00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,1.78,0.07161,E01010641,1999/00,0
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,0.76,0.44704,E01010655,1999/00,0
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,1.29,0.41032,E01010626,1999/00,0


## merge person and home

In [57]:
home_table_merge <- home_table_merge |>
    select(-cohort)

In [58]:
person_home <- person_filtered |> 
    left_join(home_table_merge, by = join_by(person_id)) 

In [59]:
head(person_home)

,person_id,birth_date,gender,CombinedEthnicity,cohort,DistCurrSch,IDACIScore_Home,LSOA11_Home,EAL
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>
1,EE91E3C77F2B7CB44F1F601317E2DE5EE76ABB0941F24D4714026B3E30FC313C,1998-09-15,M,South Asian,1998/99,0.28,0.39633,E01010830,1
2,C5553DE427E9B8F60E9B5D1E83391E1B660F415A3924B630742D52A0517F5219,1998-09-15,M,White British,1998/99,0.66,0.32669,E01012102,0
3,AC8CB4071E24E606BFFECC380A9EA11546158370916C4D8D954FA76D63FA81AE,1998-09-15,M,White British,1998/99,0.86,0.33826,E01010720,0
4,E4E84D0DEB6692712A8E875B701E097EB61235069C56626900C0951E9B112B6D,1998-09-15,M,South Asian,1998/99,0.28,0.58239,E01009366,1
5,64FA4602D88C275E27FDBD17EA4678E378D7B297D91C2CDB914150ED0CBF3785,1998-09-15,M,White British,1998/99,3.42,0.04938,E01010648,0
6,F516562D5CFDDC6CD66FB8EA18399D11BB0E2E042F569D918285FF96C3377338,1998-09-15,M,White British,1998/99,1.24,0.16338,E01023472,0


In [60]:
# save as csv
write.csv(person_home, "data/additional_person_home.csv", row.names = FALSE)

# Disability

In [61]:
student_data = "CB_2489.cb_Student"

student_table <- tbl(con, student_data) |> 
    select(person_id, Gender, Disability) |>
    filter(!is.na(person_id)) |>
    filter(!is.na(Disability))

In [62]:
student_table

# Source:   SQL [?? x 3]
# Database: BigQueryConnection
   person_id                                                   Gender Disability
   <chr>                                                       <chr>  <chr>     
 1 1BB207623557A2E675505C4CA5237B7AB9EB8EA3B3D17E70CD93A68926… F      NCOL      
 2 26E90FE4263275E646264F68FA34C104CD1AEE6BF45CF0D7F256E117A4… F      NCOL      
 3 10BC24A12C475E751290F1DD6CE0129AE5627F2676F2E7041EC4D69026… F      NCOL      
 4 EF5BDEACB889527A7BC6F6F29DC7A3CB88F77A7AF5A12E8B6DE9F88F73… F      NCOL      
 5 F7FDCCAFC13BA5A4F2FC7E407D12F14C1ACE57A0DB38874412D284C4C1… M      NCOL      
 6 D2630283F9C4DFDB64DA4509C55BA1ACC3E62B0D50FFD12CB01721F495… M      NCOL      
 7 943242326C63FCA6AAF153271971640F2CDC932543D81CB0A3E3022D93… M      NCOL      
 8 EA6F60F9824E3D26D76A78F9EC54279A433AE4390DB2953CCAB34F0300… M      NCOL      
 9 24BCE3FCC408EFDB17B492DE055E5E7E7DB013199B3768B60257B4375D… M      NCOL      
10 F27B849A12EB97ECE880026DB0E694F30594669BA0870B3DA7

In [63]:
student_df <- collect(student_table)

In [64]:
# filter person to only matching IDs
student_filtered <- student_df |>
    filter(person_id %in% person_ids$person_id)

In [65]:
head(student_filtered)

person_id,Gender,Disability
<chr>,<chr>,<chr>
F7FDCCAFC13BA5A4F2FC7E407D12F14C1ACE57A0DB38874412D284C4C1652A65,M,NCOL
E6D87B4F9F70813E09245E9B6A57989B610A678A88F2B9CECDB4616F3B1E1E98,M,NCOL
8CD319109B189BEDAA8347C66D11C90071C6F45324532E2BB5FA981456E67427,M,NCOL
30A1CEE32BBDBD1F6A973E913F42F6E35B85C3B52CAA06D68EEDA313338E79EF,M,NCOL
734096AF44FCFC666AA00CFB48B3E16A7BF1F98F8EEC2882D67381F23C20B90B,M,NCOL
91482AA2D9D0C6EE53A77B22768E757E4D64533EAFFA2812E2E824A1D1699C4A,M,NCOL


In [66]:
student_filtered |> 
  distinct(Disability) |> 
  pull(Disability)

[1] "NCOL" "NONE" "OTH"  "MED"  "LD"   "INC"  "MOB"  "BEH"  "VIS"  "HEAR"
[11] "COMM" "EAT"  "PC"   "AUT"  "HAND" "CON"

In [67]:
student_table <- student_filtered |> 
  mutate(
    SEN = case_when(
      Disability %in% c('NCOL', "", NA) ~ NA_character_,
      TRUE ~ Disability  # Handle any other unexpected cases
    )
  ) 

In [68]:
head(student_table)

person_id,Gender,Disability,SEN
<chr>,<chr>,<chr>,<chr>
F7FDCCAFC13BA5A4F2FC7E407D12F14C1ACE57A0DB38874412D284C4C1652A65,M,NCOL,NA
E6D87B4F9F70813E09245E9B6A57989B610A678A88F2B9CECDB4616F3B1E1E98,M,NCOL,NA
8CD319109B189BEDAA8347C66D11C90071C6F45324532E2BB5FA981456E67427,M,NCOL,NA
30A1CEE32BBDBD1F6A973E913F42F6E35B85C3B52CAA06D68EEDA313338E79EF,M,NCOL,NA
734096AF44FCFC666AA00CFB48B3E16A7BF1F98F8EEC2882D67381F23C20B90B,M,NCOL,NA
91482AA2D9D0C6EE53A77B22768E757E4D64533EAFFA2812E2E824A1D1699C4A,M,NCOL,NA


In [69]:
student_table |>
    group_by(SEN) |>
    tally() |>
    arrange(desc(n))

SEN,n
<chr>,<int>
NA,15623
NONE,502
LD,157
OTH,155
BEH,49
COMM,25
AUT,10
HEAR,9
MOB,9


## SEN 

In [62]:
sen_data = "CB_2489.cb_StudentEntitlementFromCensus"

sen_table <- tbl(con, sen_data) |> 
    select(person_id, SourceTable, EVERFSM_ALL, SENprovision, SENprovisionMajor, PrimarySENtype) |>
    filter(!is.na(person_id))

In [7]:
head(sen_table)

# Source:   SQL [6 x 5]
# Database: BigQueryConnection
  person_id             SourceTable EVERFSM_ALL SENprovisionMajor PrimarySENtype
  <chr>                 <chr>             <dbl> <chr>             <chr>         
1 B1D77C27358C0FAEEEFE… PLASC_2002           NA NA                NA            
2 B0F6576C4FD111B8FC7E… PLASC_2002           NA NA                NA            
3 743AE9CEC04DB16EFABD… PLASC_2002           NA NA                NA            
4 F8997374AD18D48539CC… PLASC_2002           NA NA                NA            
5 F55EE5CB232507DD90FD… PLASC_2002           NA NA                NA            
6 85F82007BB545B6BA136… PLASC_2002           NA NA                NA            

In [8]:
sen_table |> 
  distinct(SourceTable) |> 
  pull(SourceTable)

[1] "PLASC_2003"          "PLASC_2004"          "Autumn_Census_2008" 
 [4] "Autumn_Census_2011"  "Autumn_Census_2015"  "SUMMER_Census_2006" 
 [7] "Spring_Census_2007"  "Spring_Census_2008"  "Spring_Census_2010" 
[10] "Spring_Census_2011"  "Spring_Census_2012"  "Spring_Census_2013" 
[13] "Spring_Census_2014"  "Summer_Census_2007"  "Summer_Census_2009" 
[16] "Summer_Census_2011"  "Summer_Census_2014"  "Autumn_Census_2016" 
[19] "Autumn_Census_2017"  "Autumn_Census_2022"  "Spring_Census_2019" 
[22] "Spring_Census_2020"  "Spring_Census_2021"  "Spring_Census_2022" 
[25] "Summer_Census_2018"  "Summer_Census_2019"  "Summer_Census_2021" 
[28] "Summer_Census_2022"  "PLASC_2002"          "PLASC_2005"         
[31] "Autumn_Census_2007"  "Autumn_Census_2009"  "Autumn_Census_2010" 
[34] "Autumn_Census_2012"  "Autumn_Census_2013"  "Autumn_Census_2014" 
[37] "Spring_Census_2006"  "Spring_Census_2009"  "Spring_Census_2015" 
[40] "Summer_Census_2008"  "Summer_Census_2010"  "Summer_Census_2012" 
[43] "Summer_Census_2013"  "Summer_Census_2015"  "Autumn_Census_2018" 
[46] "Autumn_Census_2019"  "Autumn_Census_2020"  "Autumn_Census_2021" 
[49] "Spring_Census_2016"  "Summer_Census_2016"  "Summer_Census_20017"

Pupil Level Annual School Census (PLASC)

In [63]:
sen_df <- collect(sen_table)

In [64]:
# filter to person_ids
sen_filtered <- sen_df |>
    filter(person_id %in% person_ids$person_id)

In [65]:
# edit label for Summer_Census_20017
sen_filtered <- sen_filtered |>
    mutate(SourceTable = case_when(
        SourceTable == 'Summer_Census_20017' ~ 'Summer_Census_2017',
        TRUE ~ SourceTable)
           )

In [66]:
# extract year from source table
sen_filtered$Year <- substr(sen_filtered$SourceTable, nchar(sen_filtered$SourceTable) - 3, nchar(sen_filtered$SourceTable))

In [13]:
head(sen_filtered)

person_id,SourceTable,EVERFSM_ALL,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
330522675B341B5E7544DD15159765054E653C9A671D282531DD9657FD1493F4,PLASC_2002,NA,NA,NA,2002
BB6DCF7E9E10A7D0A39C306A6A6E608A2D5281F97B606FF116251B33ADDF78FF,PLASC_2002,NA,NA,NA,2002
9D198D6F6864728BDCDC936C82D20DC4C502630F92BF1FC5280A280BC336BBB9,PLASC_2002,NA,NA,NA,2002
F95D040681FD17D9CD3A9C5F58EB6A2EF923CB48F464CA3E70E5589705AAE398,PLASC_2002,NA,NA,NA,2002
8E57CD24611AC0B35F501586B83737B29FB1BC875EE2E44A8BA0E1E133160E50,PLASC_2003,NA,NA,NA,2003
B6DBA3DB9F059985A626C27527A3206BF277AA4261089F171F302C84FAAF20E3,PLASC_2004,NA,NA,NA,2004


In [14]:
sen_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17403


#### extract only FSM col

In [15]:
fsm <- sen_filtered |>
    select(person_id, Year, EVERFSM_ALL)

In [16]:
fsm_filtered <- fsm |>
    filter(!is.na(EVERFSM_ALL))

In [17]:
fsm_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17285


In [18]:
# keep latest year only
fsm_filtered <- fsm_filtered |>
      arrange(person_id, desc(Year)) |>
      distinct(person_id, .keep_all = TRUE)

In [19]:
fsm_filtered|>
    nrow()

[1] 17285

In [20]:
fsm_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17285


In [21]:
fsm_filtered |>
    group_by(EVERFSM_ALL) |>
    tally()

EVERFSM_ALL,n
<dbl>,<int>
0,9936
1,7349


#### merge in FSM col to main data

In [22]:
fsm_filtered <- fsm_filtered |>
    select(person_id, EVERFSM_ALL) |>
    rename(ever_fsm = EVERFSM_ALL)

In [23]:
person_home <- read.csv("data/additional_person_home.csv", header = TRUE)

In [24]:
person_home_fsm <- person_home |> 
    left_join(fsm_filtered, by = join_by(person_id)) 

In [25]:
head(person_home_fsm)

,person_id,birth_date,gender,CombinedEthnicity,cohort,DistCurrSch,IDACIScore_Home,LSOA11_Home,EAL,ever_fsm
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<int>,<dbl>
1,EE91E3C77F2B7CB44F1F601317E2DE5EE76ABB0941F24D4714026B3E30FC313C,1998-09-15,M,South Asian,1998/99,0.28,0.39633,E01010830,1,1
2,C5553DE427E9B8F60E9B5D1E83391E1B660F415A3924B630742D52A0517F5219,1998-09-15,M,White British,1998/99,0.66,0.32669,E01012102,0,1
3,AC8CB4071E24E606BFFECC380A9EA11546158370916C4D8D954FA76D63FA81AE,1998-09-15,M,White British,1998/99,0.86,0.33826,E01010720,0,1
4,E4E84D0DEB6692712A8E875B701E097EB61235069C56626900C0951E9B112B6D,1998-09-15,M,South Asian,1998/99,0.28,0.58239,E01009366,1,1
5,64FA4602D88C275E27FDBD17EA4678E378D7B297D91C2CDB914150ED0CBF3785,1998-09-15,M,White British,1998/99,3.42,0.04938,E01010648,0,0
6,F516562D5CFDDC6CD66FB8EA18399D11BB0E2E042F569D918285FF96C3377338,1998-09-15,M,White British,1998/99,1.24,0.16338,E01023472,0,0


#### SEN cols

In [67]:
sen_filtered <- sen_filtered |>
    select(-c(EVERFSM_ALL)) |>
    arrange(person_id, desc(Year))

In [27]:
head(sen_filtered)

person_id,SourceTable,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<chr>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,Autumn_Census_2018,1_NON,NA,2018
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,Summer_Census_2018,1_NON,NA,2018
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,Summer_Census_2017,1_NON,NA,2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,Autumn_Census_2017,1_NON,NA,2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,Autumn_Census_2016,1_NON,NA,2016
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,Spring_Census_2016,1_NON,NA,2016


In [28]:
sen_filtered |>
    distinct(Year) |> 
    pull(Year)

[1] "2018" "2017" "2016" "2015" "2014" "2013" "2012" "2011" "2010" "2009"
[11] "2008" "2007" "2006" "2005" "2004" "2003" "2019" "2002" "2020"

In [68]:
sen_filtered_year <- sen_filtered |>
    filter(Year %in% c('2020','2019','2018','2017','2016','2015','2014','2013','2012','2011','2010','2009','2008','2007','2006','2005'))

In [30]:
sen_filtered_year |>
    nrow()

[1] 515260

In [31]:
sen_filtered_year |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17392


In [46]:
sen_2020 <- sen_filtered_year |>
    filter(Year == '2005') |>
    filter(SENprovisionMajor != '1_NON')

In [47]:
all_2020 <- sen_filtered_year |>
    filter(Year == '2005') 

In [48]:
no2020 <- sen_filtered_year |>
    filter(Year != '2005')

In [49]:
# check for any ids in 2020 but not in previous years - if 0 then can drop 2020 (post graduation)
sen_2020_only <- anti_join(sen_2020, no2020, by = "person_id")
sen_2020_only

person_id,SourceTable,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<chr>,<chr>,<chr>


In [50]:
all_2020_only <- anti_join(all_2020, no2020, by = "person_id")
all_2020_only

person_id,SourceTable,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<chr>,<chr>,<chr>
601F0BB7480893CC5749F92DA5171829DF2D0AD6AF8FA28A3AE85BCBECFA8AEA,PLASC_2005,NA,VI,2005
740F81CFCFBB9A1610D7E64432E4D6358DF2F0BA12E8234A2056EA04392EFEEC,PLASC_2005,NA,SPLD,2005
AF04D6D1E088B6C8B9B7119D1234D5642C1741E220F8AD24FF07E45CB3318D2F,PLASC_2005,NA,NA,2005
B65FE3AC1F6360BE6BB988875A2CE44D1627E547EE7C359FDFBF525FF10773F7,PLASC_2005,NA,SLCN,2005
C280FEE80D213F9A65F7446524D6727CC0F108CED1F59D8612A14E6A68BD7B2D,PLASC_2005,NA,NA,2005


In [69]:
sen_filtered_year <- sen_filtered_year |>
    filter(Year %in% c('2018','2017','2016','2015','2014','2013','2012','2011','2010','2009','2008','2007','2006','2005'))

In [111]:
# only keep the 2020 row for one person with data only in 2020 which is no sen anyway (rather than having unknown for this id)
#sen_filtered_year <- rbind(sen_filtered_year,all_2020_only)

In [52]:
sen_filtered_year |>
    nrow()

[1] 514190

In [70]:
sen_filtered_year |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17392


In [489]:
#missing_ids <- anti_join(person_ids, sen_filtered_year, by = "person_id") %>%
#  select(person_id) %>%
#  distinct()

In [490]:
#sen_filtered |>
#    filter(person_id %in% missing_ids$person_id) |>
#    distinct(Year) |>
#    pull(Year)

In [491]:
#sen_filtered |>
#    filter(person_id %in% missing_ids$person_id) |>
#    filter(Year =='2005') |>
#    summarise(count = n_distinct(person_id))

In [492]:
#sen_filtered |>
#    filter(person_id %in% missing_ids$person_id) |>
#    filter(Year =='2005')

In [114]:
sen_filtered |>
    filter(person_id =='D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848')

person_id,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,2_SNS,NA,2011
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,2_SNS,SLCN,2011
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,N,1_NON,NA,2010
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,2_SNS,SPLD,2010
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,2_SNS,NA,2010
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,NA,NA,2009
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,N,1_NON,NA,2009
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,NA,MLD,2008
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,NA,NA,2008


In [71]:
# leave year arrange as asc so that first instance of new status remains 
sen_cut <- sen_filtered_year |>
    arrange(Year) |>
    group_by(person_id, Year) |>
    summarise(across(everything(), ~ ifelse(all(is.na(.)), NA, na.omit(.)[1])), .groups = "drop")

In [72]:
sen_cut |>
    nrow()

[1] 202762

In [73]:
sen_cut |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17392


#### clean SEN cols

In [74]:
sen_cut |> 
    nrow()

[1] 202762

In [75]:
sen_clean <- sen_cut 

In [76]:
sen_clean |>
    group_by(SENprovision) |>
    tally() |>
    arrange(desc(n))

SENprovision,n
<chr>,<int>
N,159097
A,24676
P,11440
S,4175
K,2980
E,394


In [77]:
sen_clean <- sen_clean |>
    mutate(SENprovisionNEW = case_when(
        SENprovision %in% c('N') ~ 'No SEN',
        SENprovision %in% c('A','P','K') ~ 'SEN support',
        SENprovision %in% c('S','E') ~ 'EHC plan',
        TRUE ~ NA
            )
         )

In [78]:
head(sen_clean)

person_id,Year,SourceTable,SENprovision,SENprovisionMajor,PrimarySENtype,SENprovisionNEW
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2005,PLASC_2005,N,NA,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2006,Spring_Census_2006,N,NA,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2007,Summer_Census_2007,N,NA,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2008,Spring_Census_2008,N,NA,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2009,Spring_Census_2009,N,1_NON,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2010,Summer_Census_2010,N,1_NON,NA,No SEN


In [79]:
sen_clean |>
    group_by(SENprovisionMajor) |>
    tally() 

SENprovisionMajor,n
<chr>,<int>
1_NON,108128
2_SNS,27545
3_SS,3646
NA,63443


In [80]:
sen_clean <- sen_clean |>
    select(-SENprovision) |>
    mutate(SENmajorNEW = case_when(
        SENprovisionMajor == '1_NON' ~ 'No SEN',
        SENprovisionMajor == '2_SNS' ~ 'SEN support',
        SENprovisionMajor == '3_SS' ~ 'EHC plan',
        TRUE ~ NA
            )
         )

In [81]:
head(sen_clean)

person_id,Year,SourceTable,SENprovisionMajor,PrimarySENtype,SENprovisionNEW,SENmajorNEW
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2005,PLASC_2005,NA,NA,No SEN,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2006,Spring_Census_2006,NA,NA,No SEN,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2007,Summer_Census_2007,NA,NA,No SEN,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2008,Spring_Census_2008,NA,NA,No SEN,NA
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2009,Spring_Census_2009,1_NON,NA,No SEN,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2010,Summer_Census_2010,1_NON,NA,No SEN,No SEN


In [82]:
# check where two new cols dont match
sen_clean |>
    filter(SENprovisionNEW != SENmajorNEW & !is.na(SENprovisionNEW) & !is.na(SENmajorNEW))

person_id,Year,SourceTable,SENprovisionMajor,PrimarySENtype,SENprovisionNEW,SENmajorNEW
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0037E941FE88C2503AF508B5306803DB1693F3200533648162692820D1EA221D,2009,Autumn_Census_2009,1_NON,NA,SEN support,No SEN
00DF0E4C51A21FCBAB2877360CF6663CE4D6A38CDEF3079601A319F3B250AEC6,2009,Autumn_Census_2009,2_SNS,NA,No SEN,SEN support
0122DBAEF5317FC0247DB65A54E5AF7ECE614F9651BAF9A1C2948D0E84FAC7D5,2009,Autumn_Census_2009,1_NON,NA,SEN support,No SEN
01846FB39CA12D92AC66CEA2227079A681F9093AFB2925A7B7578AB5B44245A3,2009,Autumn_Census_2009,1_NON,NA,SEN support,No SEN
0377438AD33436A7D103C1FBE4E1FDBF06466C75946C3E5227494FCA5D75A28E,2009,Autumn_Census_2009,2_SNS,NA,No SEN,SEN support
03D7BD9ED69833F2058E65F946BEFC23B595C608B3D3CCE833ECAA4D89136DB9,2009,Autumn_Census_2009,1_NON,NA,SEN support,No SEN
040C0FFF4ADA6DFFC51317E106CEA7DC7CE97C71B0E0DCC837D5C57BC079875E,2009,Autumn_Census_2009,1_NON,NA,SEN support,No SEN
04B8154970F2C510741B4D0B1E2DA92E745CE931E114BB151A670FF04CBABACF,2009,Autumn_Census_2009,1_NON,NA,SEN support,No SEN
056F5A1A5ECE89A2FA652093AFADA7C163CE1F6574469191F1940804EC11A171,2009,Autumn_Census_2009,2_SNS,NA,No SEN,SEN support


In [83]:
# merge values across two new cols
# where one of the new cols is no sen and the other is sen, keep sen
sen_clean <- sen_clean |>
    mutate(SENlevel = case_when(
        SENprovisionNEW == SENmajorNEW ~ SENprovisionNEW,  # both are the same
        is.na(SENprovisionNEW) & !is.na(SENmajorNEW) ~ SENmajorNEW,  # drop NA
        !is.na(SENprovisionNEW) & is.na(SENmajorNEW) ~ SENprovisionNEW,  # drop NA
        SENprovisionNEW == "No SEN" & SENmajorNEW == "SEN support" ~ SENmajorNEW,  # SEN support over No SEN
        SENprovisionNEW == "SEN support" & SENmajorNEW == "No SEN" ~ SENprovisionNEW,  # SEN support over No SEN
        SENprovisionNEW == "SEN support" & SENmajorNEW == "EHC plan" ~ SENmajorNEW,  # EHC plan over SEN support 
        SENprovisionNEW == "EHC plan" & SENmajorNEW == "SEN support" ~ SENprovisionNEW,  # EHC plan over SEN support 
        SENprovisionNEW == "No SEN" & SENmajorNEW == "EHC plan" ~ SENmajorNEW,  # EHC plan over SEN support 
        TRUE ~ NA
  ))

In [84]:
# check where two new cols dont match incase of EHCP col not matching
sen_clean |>
    filter(is.na(SENlevel))

person_id,Year,SourceTable,SENprovisionMajor,PrimarySENtype,SENprovisionNEW,SENmajorNEW,SENlevel
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [85]:
sen_clean <- sen_clean |>
    select(-SENprovisionMajor)

In [86]:
head(sen_clean)

person_id,Year,SourceTable,PrimarySENtype,SENprovisionNEW,SENmajorNEW,SENlevel
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2005,PLASC_2005,NA,No SEN,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2006,Spring_Census_2006,NA,No SEN,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2007,Summer_Census_2007,NA,No SEN,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2008,Spring_Census_2008,NA,No SEN,NA,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2009,Spring_Census_2009,NA,No SEN,No SEN,No SEN
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2010,Summer_Census_2010,NA,No SEN,No SEN,No SEN


In [87]:
sen_clean <- sen_clean |>
    select(-c(SENprovisionNEW,SENmajorNEW))

In [89]:
sen_clean <- sen_clean |>
    select(-SourceTable)

In [90]:
# copy previous value where sen level is the same but type is NA
sen_clean <- sen_clean |>
    arrange(person_id, Year) |>  
    group_by(person_id) |>
    mutate(PrimarySENtype = case_when(
          # if NA and SENlevel is same, and prev not NA take prev value
          is.na(PrimarySENtype) & lag(SENlevel) == SENlevel & !is.na(lag(PrimarySENtype)) ~ lag(PrimarySENtype),  
          TRUE ~ PrimarySENtype  # Keep original value otherwise
        )) |>
    #fill(PrimarySENtype, .direction = "down") |>
    ungroup()

In [91]:
# repeat
sen_clean <- sen_clean |>
    arrange(person_id, Year) |>  
    group_by(person_id) |>
    mutate(PrimarySENtype = case_when(
          # if NA and SENlevel is same, and prev not NA take prev value
          is.na(PrimarySENtype) & lag(SENlevel) == SENlevel & !is.na(lag(PrimarySENtype)) ~ lag(PrimarySENtype),  
          TRUE ~ PrimarySENtype  # Keep original value otherwise
        )) |>
    #fill(PrimarySENtype, .direction = "down") |>
    ungroup()

What are the implications of having an SEN but it only being identified / support being put in place when in year 11???

In [92]:
sen_clean |>
    filter(SENlevel != 'No SEN')

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2009,NA,SEN support
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2010,NA,SEN support
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2008,NA,SEN support
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2007,NA,SEN support
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2008,NA,SEN support
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2009,NA,SEN support
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2010,NA,SEN support
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2011,NA,SEN support
001F42538A9E6E496F13CD05B5E9FE91EB3B20F0CEBECA1B76CC5E7151B7A0D4,2011,NA,SEN support


In [93]:
sen_clean <- sen_clean |>
    mutate(PrimarySENtype = case_when(
        PrimarySENtype == 'BESD' ~ 'SEMH',
        TRUE ~ PrimarySENtype)
           )

In [ ]:
sen_clean <- sen_clean |>
    

#### join cohort id

In [94]:
sen_clean_cohort <- sen_clean |>
    left_join(person_ids, by = join_by(person_id)) 

Warning message in left_join(sen_clean, person_ids, by = join_by(person_id)):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 53 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


In [95]:
sen_clean_cohort

person_id,Year,PrimarySENtype,SENlevel,NCCIS_ACADYR
<chr>,<chr>,<chr>,<chr>,<chr>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2005,NA,No SEN,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2006,NA,No SEN,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2007,NA,No SEN,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2008,NA,No SEN,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2009,NA,No SEN,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2010,NA,No SEN,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2011,NA,No SEN,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2012,NA,No SEN,2016/2017
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,2013,NA,No SEN,2016/2017


#### check missing sen type against extras from exclusion tables

In [96]:
sen <- read.csv('data/additional_sen_extras.csv', header = TRUE)

In [97]:
head(sen)

,person_id,PrimarySENtype
,<chr>,<chr>
1,13A5184415FCC22BFF3F2EFFD8D3B69FAB1949AFB346B8A15B449CBED2CDB473,BESD
2,02304796B373267AE442202D65322A36B8844DD043A88734014D6434DD7D1F12,BESD
3,04584E57ADBD062B2BA9DB9076CF692E45F5767837BD87EEE4FED6DB521BB84B,OTH
4,66927CC16F370C097C232D8581FB0E92E89F688DE966B50CD1A128B631F9DB6A,MLD
5,778E21A7061D57C78E5986180FAE5D061F1F8B2B69F1925CDD0FCDC4B424AEF1,BESD
6,7D46116A75B15250162ECE83B71941167D4FFAE82D39C508681D25F69C50BF32,BESD


In [98]:
missing_type <- sen_clean_cohort |>
    filter(SENlevel != 'No SEN' & is.na(PrimarySENtype))

In [99]:
missing_type

person_id,Year,PrimarySENtype,SENlevel,NCCIS_ACADYR
<chr>,<chr>,<chr>,<chr>,<chr>
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2009,NA,SEN support,2015/2016
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,2010,NA,SEN support,2015/2016
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2008,NA,SEN support,2016/2017
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,2008,NA,SEN support,2016/2017
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2007,NA,SEN support,2016/2017
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2008,NA,SEN support,2016/2017
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2009,NA,SEN support,2016/2017
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2010,NA,SEN support,2016/2017
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,2011,NA,SEN support,2016/2017


In [100]:
# Check if all person_ids in person_id are in person
any(sen$person_id %in% missing_type$person_id)

[1] TRUE

In [101]:
sum(sen$person_id %in% missing_type$person_id)

[1] 805

In [102]:
sen <- sen |>
    mutate(PrimarySENtype = case_when(
        PrimarySENtype == 'BESD' ~ 'SEMH',
        TRUE ~ PrimarySENtype)
           )

#### compute 'ever' values

In [103]:
sen_clean_cohort <- sen_clean_cohort |> 
  mutate(Year = as.integer(Year))  # Convert Year to integer

In [104]:
sen_final <- sen_clean_cohort |> 
  group_by(person_id) |> 
  summarize(
    ever_SEN_support = any(SENlevel == "SEN support"),
    first_SEN_support_year = ifelse(ever_SEN_support, min(Year[SENlevel == "SEN support"], na.rm = TRUE), NA),
    SEN_support_need = ifelse(ever_SEN_support, paste(na.omit(unique(PrimarySENtype[SENlevel == "SEN support"])), collapse = ", "), NA),

    ever_EHC_plan = any(SENlevel == "EHC plan"),
    first_EHC_plan_year = ifelse(ever_EHC_plan, min(Year[SENlevel == "EHC plan"], na.rm = TRUE), NA),
    EHC_plan_need = ifelse(ever_EHC_plan, paste(na.omit(unique(PrimarySENtype[SENlevel == "EHC plan"])), collapse = ", "), NA),

    no_SEN = all(SENlevel == "No SEN")
  )

In [105]:
sen_final

person_id,ever_SEN_support,first_SEN_support_year,SEN_support_need,ever_EHC_plan,first_EHC_plan_year,EHC_plan_need,no_SEN
<chr>,<lgl>,<int>,<chr>,<lgl>,<int>,<chr>,<lgl>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,FALSE,NA,NA,FALSE,NA,NA,TRUE
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,TRUE,2009,,FALSE,NA,NA,FALSE
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,FALSE,NA,NA,FALSE,NA,NA,TRUE
00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,FALSE,NA,NA,FALSE,NA,NA,TRUE
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,TRUE,2008,,FALSE,NA,NA,FALSE
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,TRUE,2007,,FALSE,NA,NA,FALSE
001F42538A9E6E496F13CD05B5E9FE91EB3B20F0CEBECA1B76CC5E7151B7A0D4,TRUE,2011,,FALSE,NA,NA,FALSE
00211975518FD5336E43850D59B5F5DDA7CD98B212760111CB40528BDECF3B1B,TRUE,2008,"SPLD, SEMH",TRUE,2014,SEMH,FALSE
00237E9C21696BB29409630166F2D9CCEC24B83FB120AEDD878DDD54C6516B3D,FALSE,NA,NA,FALSE,NA,NA,TRUE


In [106]:
sen_final |> nrow()

[1] 17392

#### ARCHIVE filter to one record pp with year as first record of SEN need

In [58]:
# group by all cols except year, and keep first row of duplicates
sen_merge <- sen_clean |>
    arrange(desc(Year)) |>
    group_by(across(-Year)) |>
    filter(row_number() == 1) |>
    ungroup() |>
    arrange(person_id)

In [59]:
sen_merge |>
    filter(person_id == 'D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E')

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2020,NA,No SEN
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2017,SPLD,SEN support
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2014,NA,SEN support


In [60]:
sen_merge |>
arrange(person_id, desc(Year))

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017,ASD,SEN support
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2015,NA,No SEN
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2017,MLD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2006,NA,No SEN
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,NA,EHC plan
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2017,MLD,EHC plan
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2011,MLD,SEN support
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2008,NA,No SEN
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2005,SLCN,SEN support


In [575]:
# copy previous value where sen level is NA but type not NA
sen_level_clean <- sen_merge |>
    arrange(person_id, Year) |>  
    group_by(person_id) |>
    mutate(SENlevel = case_when(
          # if NA and prev SENlevel is SEN support, and type not NA 
          SENlevel == 'No SEN' & lag(SENlevel) == 'SEN support' & !is.na(PrimarySENtype) ~ 'SEN support',  
          SENlevel == 'No SEN' & lag(SENlevel) == 'EHC plan' & !is.na(PrimarySENtype) ~ 'EHC plan',
          TRUE ~ SENlevel  # Keep original value otherwise
        )) |>
    ungroup()

In [576]:
sen_level_clean |>
    filter(person_id == 'D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E')

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2007,NA,No SEN
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2008,NA,SEN support
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2010,SPLD,SEN support
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2013,SPLD,SEN support


In [585]:
# repeat but for desc year
sen_level_desc <- sen_level_clean |>
    arrange(person_id, desc(Year)) |>  
    group_by(person_id) |>
    mutate(SENlevel = case_when(
          # if NA and prev SENlevel is SEN support, and type not NA 
          SENlevel == 'No SEN' & lag(SENlevel) == 'SEN support' & !is.na(PrimarySENtype) ~ 'SEN support',  
          SENlevel == 'No SEN' & lag(SENlevel) == 'EHC plan' & !is.na(PrimarySENtype) ~ 'EHC plan',
          TRUE ~ SENlevel  # Keep original value otherwise
        )) |>
    ungroup()

In [591]:
# repeat to catch gaps
sen_level_desc <- sen_level_desc |>
    arrange(person_id, Year) |>  
    group_by(person_id) |>
    mutate(SENlevel = case_when(
          # if NA and prev SENlevel is SEN support, and type not NA 
          SENlevel == 'No SEN' & lag(SENlevel) == 'SEN support' & !is.na(PrimarySENtype) ~ 'SEN support',  
          SENlevel == 'No SEN' & lag(SENlevel) == 'EHC plan' & !is.na(PrimarySENtype) ~ 'EHC plan',
          TRUE ~ SENlevel  # Keep original value otherwise
        )) |>
    ungroup()

In [594]:
sen_level_desc |>
    filter(person_id == 'D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147')

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147,2006,NA,No SEN
D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147,2008,SEMH,SEN support
D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147,2013,SEMH,SEN support
D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147,2016,SEMH,SEN support


In [595]:
missing_senlevel <- sen_level_desc |>
    filter(SENlevel == 'No SEN' & !is.na(PrimarySENtype))

In [596]:
missing_senlevel |>
    summarize(n_distinct(person_id))

n_distinct(person_id)
<int>
64


In [598]:
sen_level_desc |>
    filter(person_id %in% missing_senlevel$person_id) |>
    distinct(SENlevel)

SENlevel
<chr>
No SEN


In [599]:
sen_level_desc |>
    filter(person_id %in% missing_senlevel$person_id) |>
    arrange(person_id, Year)

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
05BECC2FA9168CC3CC4A8669F2B6BEF1235BB7A7386DDA71141D0444C1A34D87,2005,OTH,No SEN
0607C1DA04A1A88F14F80F5DB04C9112B950A68544F2F21E241F94F3AC7D6E78,2006,NA,No SEN
0607C1DA04A1A88F14F80F5DB04C9112B950A68544F2F21E241F94F3AC7D6E78,2015,OTH,No SEN
137AAFFB957B50C9C3509763EEE31B4AD73661BB93279CBBCCFB377BDE975C8D,2006,NA,No SEN
137AAFFB957B50C9C3509763EEE31B4AD73661BB93279CBBCCFB377BDE975C8D,2015,SPLD,No SEN
1E2DDA056F48950C6617E5D8209A6CB6E5583CD02031694903C9E11461C1B70B,2006,NA,No SEN
1E2DDA056F48950C6617E5D8209A6CB6E5583CD02031694903C9E11461C1B70B,2015,OTH,No SEN
20E40E9631F550A80DC84266D8EA9118EE12F0C0498AEF95D4ED1DAC9A5124ED,2006,NA,No SEN
20E40E9631F550A80DC84266D8EA9118EE12F0C0498AEF95D4ED1DAC9A5124ED,2015,SPLD,No SEN


In [603]:
sen_final <- sen_level_desc |>
    arrange(desc(Year)) |>            
    group_by(person_id) |>            
    filter(row_number() == 1) |>      
    ungroup()   

In [604]:
sen_final |>
    group_by(PrimarySENtype) |>
    tally() 

PrimarySENtype,n
<chr>,<int>
ASD,230
HI,128
MLD,1540
MSI,3
NSA,54
OTH,204
PD,150
PMLD,27
SEMH,1128


In [606]:
head(sen_final)

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
0BAA7959F866DC200B0DE7C16B5B16DD63D6D6BA086D46BA9D26025AC7FB1FB6,2020,OTH,SEN support
18A1DBCA368AFAADA2DAC40BD35DE4E1756E5C5905C643A9DFD8590ED65BC980,2020,HI,SEN support
18BDC8758A1C2629E1B3FE7677D649961D106A4E3C5371B14B2EE6FBC12D6162,2020,VI,SEN support
191B614E755130087F1A4C6F61D7CA413FC57CDE42DE350168AF9F21079A5A3D,2020,VI,SEN support
1E8EDACA982683915764BFB3DA811D133E1AD9879F0512C019E4307EFF8E2D13,2020,SEMH,SEN support
248B0C2E0DD811AC20D8D8940381A477C3041C6B299278C2F7C44AB3E3E098A0,2020,OTH,SEN support


In [621]:
sen_final <- sen_final |>
    rename(SENtype = PrimarySENtype) |>
    select(-Year)

In [622]:
sen_final |>
    nrow()

[1] 17224

In [623]:
sen_final |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


#### merge SEN into main data

In [107]:
# save as csv
write.csv(sen_final, "data/additional_sen_final.csv", row.names = FALSE)

In [108]:
person_home_fsm_sen <- sen_final |> 
    left_join(person_home_fsm, by = join_by(person_id)) 

In [109]:
person_home_fsm_sen

person_id,ever_SEN_support,first_SEN_support_year,SEN_support_need,ever_EHC_plan,first_EHC_plan_year,EHC_plan_need,no_SEN,birth_date,gender,CombinedEthnicity,cohort,DistCurrSch,IDACIScore_Home,LSOA11_Home,EAL,ever_fsm
<chr>,<lgl>,<int>,<chr>,<lgl>,<int>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<int>,<dbl>
0002FD06417A994A32C6C24D7236F7C6D98ACD309E999386EFE3A04741BF3433,FALSE,NA,NA,FALSE,NA,NA,TRUE,1999-09-15,F,White British,1999/00,2.45,0.04778,E01027555,0,0
00063B7DE8D61E07E54571351684A37504385EBD55879708236F3579C4E33661,TRUE,2009,,FALSE,NA,NA,FALSE,1999-08-15,F,White British,1998/99,4.71,0.35519,E01010703,0,0
000709767CB306493FC46DAC5E2E6B92D691A0C8A5299139ACA3871DAE72A909,FALSE,NA,NA,FALSE,NA,NA,TRUE,1998-11-15,M,South Asian,1998/99,0.33,0.37575,E01010742,0,0
00131B1E9F8FD120B0E688913FCFE2F0C27F1B2C6584AE0378B38B75E74B10AC,FALSE,NA,NA,FALSE,NA,NA,TRUE,2000-05-15,M,White British,1999/00,1.78,0.07161,E01010641,0,0
0015AFB059DFAA539E96CA31373F8B3E2244BBE7737F72CD606C785A47AA862D,TRUE,2008,,FALSE,NA,NA,FALSE,2000-08-15,F,White British,1999/00,0.76,0.44704,E01010655,0,1
00183D5045A271DEF4B6CC51797F891551229B01E913B4C0619CCBC638EA63E7,TRUE,2007,,FALSE,NA,NA,FALSE,1999-09-15,M,South Asian,1999/00,1.29,0.41032,E01010626,0,1
001F42538A9E6E496F13CD05B5E9FE91EB3B20F0CEBECA1B76CC5E7151B7A0D4,TRUE,2011,,FALSE,NA,NA,FALSE,1999-01-15,F,South Asian,1998/99,1.37,0.06820,E01010568,0,1
00211975518FD5336E43850D59B5F5DDA7CD98B212760111CB40528BDECF3B1B,TRUE,2008,"SPLD, SEMH",TRUE,2014,SEMH,FALSE,1998-10-15,M,White British,1998/99,4.27,0.13373,E01010752,0,1
00237E9C21696BB29409630166F2D9CCEC24B83FB120AEDD878DDD54C6516B3D,FALSE,NA,NA,FALSE,NA,NA,TRUE,1999-03-15,M,White British,1998/99,0.10,0.03718,E01027577,0,0


In [110]:
# save as csv
write.csv(person_home_fsm_sen, "data/additional_person_home_fsm_sen.csv", row.names = FALSE)